# Datamine CONPOL Process Example

This notebook demonstrates how to configure and run the **`conpol`** process wrapper in `dmstudio`.

## Process Description

## CONPOL

Create a convex polygon around an X,Y point data set.

To prevent large areas within the polygon having no data, a maximum chord length can be specified for the perimeter, so that the perimeter clings to the limits of the data creating a concave perimeter. The perimeter is established wih a PVALUE of 1.0, and a ZP value of 0.0. The standard perimeter file that is output will only have one perimeter.

### Input Files:

* **in** (Undefined):
  Input file.
  Required=Yes

### Output Files:

* **perimout** (String):
  Output Perimeter file.
  Required=Yes

### Fields:

* **x** (Numeric : IN):
  X coordinate field on input file.
  Default=Undefined
  Required=Yes

* **y** (Numeric : IN):
  Y coordinate field on input file.
  Default=Undefined
  Required=Yes

### Parameters:

* **maxlen**:
  Maximum chord length, which should be greater than the distance between separate groups
  of data if all groups are to be included in the one perimeter and greater than the
  average point separation.
  Range=
  Values=
  Default=
  Required=No

* **extdis**:
  Distance to extend the perimeter (0).
  Range=
  Values=
  Default=
  Required=No

* **checkdup**:
  Check for duplicate points in the input file. If this is turned on (set to 1) then
  processing takes much longer. For data sets with more than a few thousand points it is
  recommended that this is turned off (set to 1). Duplicate points can be removed from the
  input data file by using the FILTPO process before running CONPOL. =0 : Do not check for
  duplicate points - processing is much faster. =1 : Check for duplicate points.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('conpol')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute conpol
print("Running conpol...")
dm_cmd.conpol(
    in_i='t_assays',  # required
    perimout_o='t_conpol_out',  # required
    x_f='X',  # required
    y_f='Y',  # required
    # maxlen_p='optional',  # optional
    # extdis_p='optional',  # optional
    # checkdup_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("conpol execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_conpol_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")